# SMĀRANA cloud training — synthetic DDA demo

Repository: https://gitlab.com/smarana-group1/smarana/-/tree/v2

This self-contained notebook embeds the corrected feature builder and training scripts from commit `5d6dff35bbff0a4792e15137b620b07e04bdb62d`. No credentials or patient records are uploaded. It trains a Random Forest on **simulated sessions**, evaluates on completely held-out simulated users, and exports a deployment-compatible artifact. Scores measure imitation of the simulation policy; they do not establish clinical efficacy. CPU runtime is sufficient.


In [ ]:
%pip install -q numpy==2.1.3 pandas==2.2.3 scikit-learn==1.6.1 joblib==1.4.2


In [ ]:
from pathlib import Path
import json, os, hashlib, sys, subprocess
ROOT = Path("/content/smarana-dda-demo")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
SOURCES = {'backend/apps/games/notebook_dda.py': '"""Production boundary for the notebook\'s supervised RF pipeline.\n\nNo training, pandas history construction, or notebook execution happens at runtime.\nArtifacts must carry the corrected feature contract and a training-only RT median.\n"""\n\nimport importlib\nimport logging\nimport math\nfrom pathlib import Path\nfrom typing import Any\n\nlogger = logging.getLogger(__name__)\nFEATURE_COLUMNS = [\n    "current_difficulty",\n    "accuracy",\n    "response_time",\n    "hints_used",\n    "early_exit",\n    "rounds_this_session",\n    "session_duration",\n    "rolling_accuracy_3",\n    "rolling_response_time_3",\n    "accuracy_trend",\n    "response_time_trend",\n    "recent_hint_rate",\n    "consecutive_errors",\n    "consecutive_successes",\n    "previous_adjustment",\n    "difficulty_change_count_5",\n    "recent_early_exit_rate",\n    "is_cold_start",\n    "condition",\n]\nFEATURE_CONTRACT = "smarana-history-v1"\n\n\ndef build_features(\n    current: dict[str, Any], history: list[dict[str, Any]], rt_neutral: float, condition: str\n) -> dict[str, Any]:\n    """History oldest first, excludes current. Time units are seconds, not ms."""\n    if condition not in {"none", "low_vision", "motor_tremor"}:\n        raise ValueError("Accessibility context has not been collected")\n    if not math.isfinite(rt_neutral) or rt_neutral <= 0:\n        raise ValueError("Missing training RT baseline")\n    past = history[-6:]\n    recent = past[-3:]\n    enough = len(recent) == 3\n\n    def mean(rows: list[dict[str, Any]], key: str) -> float:\n        return float(sum(row[key] for row in rows) / len(rows))\n\n    acc = mean(recent, "accuracy") if enough else 0.65\n    rt = mean(recent, "response_time") if enough else rt_neutral\n    changes = [row.get("adjustment", 0) for row in history[-5:]]\n\n    def streak(predicate: Any) -> int:\n        count = 0\n        for row in reversed(history):\n            if not predicate(row["accuracy"]):\n                break\n            count += 1\n        return count\n\n    result = {\n        "current_difficulty": 0.1 + (current["difficulty"] - 1) * 0.225,\n        "accuracy": current["accuracy"],\n        "response_time": current["reaction_time_ms"] / 1000,\n        "hints_used": current["hints_used"],\n        "early_exit": int(current["early_exit"]),\n        "rounds_this_session": current["rounds_completed"],\n        "session_duration": current["session_duration_sec"],\n        "rolling_accuracy_3": acc,\n        "rolling_response_time_3": rt,\n        "accuracy_trend": acc - mean(past[:3], "accuracy") if len(past) == 6 else 0,\n        "response_time_trend": rt - mean(past[:3], "response_time") if len(past) == 6 else 0,\n        "recent_hint_rate": sum(row["hints_used"] for row in recent)\n        / max(1, sum(row["rounds"] for row in recent))\n        if enough\n        else 0,\n        "consecutive_errors": streak(lambda accuracy: accuracy < 0.55),\n        "consecutive_successes": streak(lambda accuracy: accuracy > 0.75),\n        "previous_adjustment": changes[-1] if changes else 0,\n        "difficulty_change_count_5": sum(value != 0 for value in changes)\n        if len(changes) == 5\n        else 0,\n        "recent_early_exit_rate": sum(row["early_exit"] for row in history[-5:]) / 5\n        if len(history) >= 5\n        else 0,\n        "is_cold_start": int(len(history) < 3),\n        "condition": condition,\n    }\n    if any(not math.isfinite(value) for key, value in result.items() if key != "condition"):\n        raise ValueError("Non-finite feature")\n    return {key: result[key] for key in FEATURE_COLUMNS}\n\n\ndef recommend(\n    artifact_path: str, current: dict[str, Any], history: list[dict[str, Any]], condition: str\n) -> dict[str, Any]:\n    held = {"adjustment": 0, "engine_version": "notebook-rf-v1", "reason": "model_unavailable"}\n    if len(history) < 3:\n        return {**held, "reason": "cold_start"}\n    try:\n        # Only a deployment-controlled file is accepted; never a request-supplied pickle.\n        joblib = importlib.import_module("joblib")\n        pd = importlib.import_module("pandas")\n        sklearn = importlib.import_module("sklearn")\n\n        artifact = joblib.load(Path(artifact_path))\n        metadata = artifact["metadata"]\n        if (\n            metadata["feature_columns"] != FEATURE_COLUMNS\n            or metadata.get("feature_contract") != FEATURE_CONTRACT\n            or metadata["library_versions"]["sklearn"] != sklearn.__version__\n        ):\n            raise ValueError("Artifact feature/library version mismatch")\n        features = build_features(current, history, metadata["rt_neutral"], condition)\n        schema = metadata["feature_schema"]\n        for key in FEATURE_COLUMNS[:-1]:\n            spec = schema[key]\n            if not spec["min"] <= features[key] <= spec["max"]:\n                raise ValueError(f"Out-of-domain feature: {key}")\n        pipeline = artifact["pipeline"]\n        confidence = float(\n            max(pipeline.predict_proba(pd.DataFrame([features], columns=FEATURE_COLUMNS))[0])\n        )\n        prediction = pipeline.predict(pd.DataFrame([features], columns=FEATURE_COLUMNS))[0]\n        if prediction not in {-1, 0, 1}:\n            raise ValueError("Invalid model adjustment")\n        adjustment = int(prediction) if confidence >= 0.6 else 0\n        # Conservative replacement for the notebook\'s undefined safeguard function.\n        if current["early_exit"] or current["accuracy"] < 0.55:\n            adjustment = -1\n        if adjustment > 0 and (\n            features["consecutive_errors"] > 0\n            or features["recent_early_exit_rate"] > 0.2\n            or current["errors"] >= max(1, current["rounds_completed"] / 2)\n        ):\n            adjustment = 0\n        # Notebook rate limits: 4 changes/10 sessions, 3-session cooldown after\n        # increase, 1 after decrease, two consecutive increase signals.\n        flags = [row.get("adjustment", 0) for row in history[-10:]]\n        last_change = next((index for index, value in enumerate(reversed(flags)) if value), None)\n        if sum(value != 0 for value in flags) >= 4:\n            adjustment = 0\n        elif adjustment == 1 and (last_change is not None and last_change < 3):\n            adjustment = 0\n        elif adjustment == 1 and history[-1].get("raw_prediction") != 1:\n            adjustment = 0\n        elif adjustment == -1 and last_change == 0 and not current["early_exit"]:\n            adjustment = 0\n        return {\n            "adjustment": adjustment,\n            "raw_prediction": int(prediction),\n            "engine_version": "notebook-rf-v1",\n            "model_version": metadata["model_version"],\n            "reason": "model",\n        }\n    except Exception:\n        logger.exception("Notebook DDA inference held difficulty")\n        return held\n\n\ndef recommend_for_patient(\n    patient: Any, game: Any, current: dict[str, Any], session_id: Any = None\n) -> dict[str, Any]:\n    from django.conf import settings\n\n    from apps.games.models import GameSession\n\n    sessions = GameSession.objects.filter(patient=patient, game=game, guest_mode=False)\n    if session_id:\n        sessions = sessions.exclude(id=session_id)\n    history = []\n    for session in sessions.order_by("ended_at", "id"):\n        change = session.difficulty_changes.first()\n        history.append(\n            {\n                "accuracy": session.metrics["accuracy"],\n                "response_time": session.metrics["mean_reaction_ms"] / 1000,\n                "hints_used": session.metrics["hints_used"],\n                "rounds": session.metrics["rounds"],\n                "early_exit": not session.metrics["completed"],\n                "raw_prediction": session.metrics.get("dda", {}).get("raw_prediction", 0),\n                "adjustment": change.to_level - change.from_level if change else 0,\n            }\n        )\n    return recommend(\n        settings.DDA_MODEL_ARTIFACT,\n        current,\n        history,\n        patient.accessibility.get("dda_condition", ""),\n    )\n', 'scripts/generate-synthetic-dda.py': '#!/usr/bin/env python3\n"""Generate reproducible, non-clinical DDA training rows for a demo model."""\n\nimport argparse\nimport json\nimport random\nfrom datetime import UTC, datetime, timedelta\nfrom pathlib import Path\n\nGAMES = (\n    "sequence_recall", "memory_match", "find_the_change", "object_sorting",\n    "daily_routine", "word_recall", "visual_search", "pattern_completion",\n    "spatial_recall", "attention_tap", "association_game", "personal_memory",\n)\nCONDITIONS = ("none", "none", "none", "low_vision", "motor_tremor")\n\n\ndef clipped(value: float, low: float, high: float) -> float:\n    return max(low, min(high, value))\n\n\ndef adjustment(history: list[dict[str, object]], accuracy: float, early_exit: bool) -> int:\n    """A transparent simulated care-policy label, never a clinical label."""\n    if early_exit or accuracy < 0.55:\n        return -1\n    recent = history[-3:]\n    if len(recent) == 3 and all(float(row["accuracy"]) >= 0.85 for row in recent) and accuracy >= 0.85:\n        return 1\n    return 0\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("output", type=Path)\n    parser.add_argument("--patients", type=int, default=120)\n    parser.add_argument("--sessions-per-game", type=int, default=16)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if args.patients < 10 or args.sessions_per_game < 6:\n        raise ValueError("Use at least 10 patients and 6 sessions per game.")\n\n    rng = random.Random(args.seed)\n    rows: list[dict[str, object]] = []\n    origin = datetime(2026, 1, 1, tzinfo=UTC)\n    for patient_index in range(args.patients):\n        ability = rng.uniform(-0.25, 0.25)\n        condition = rng.choice(CONDITIONS)\n        for game_index, game_id in enumerate(GAMES):\n            level = rng.randint(1, 3)\n            history: list[dict[str, object]] = []\n            for session_index in range(args.sessions_per_game):\n                load = (level - 1) * 0.11\n                fatigue = max(0, session_index - 10) * 0.012\n                accuracy = clipped(0.77 + ability - load - fatigue + rng.gauss(0, 0.11), 0.05, 1.0)\n                early_exit = accuracy < 0.42 or rng.random() < (0.015 + fatigue / 3)\n                rounds = rng.randint(3, 6)\n                hints = max(0, int(round((1 - accuracy) * rounds + rng.gauss(0, 0.6))))\n                reaction_ms = int(clipped(1500 + level * 380 - ability * 800 + fatigue * 1500 + rng.gauss(0, 260), 500, 6000))\n                event = {\n                    "difficulty": level,\n                    "accuracy": round(accuracy, 4),\n                    "reaction_time_ms": reaction_ms,\n                    "hints_used": hints,\n                    "rounds_completed": 0 if early_exit else rounds,\n                    "early_exit": early_exit,\n                    "session_duration_sec": int(clipped(rounds * reaction_ms / 1000 * 1.8, 20, 600)),\n                    "errors": max(0, rounds - int(round(accuracy * rounds))),\n                }\n                target = adjustment(history, accuracy, early_exit)\n                next_level = int(clipped(level + target, 1, 5))\n                rows.append({\n                    "patient_id": f"synthetic-{patient_index:03d}",\n                    "game_id": game_id,\n                    "timestamp": (origin + timedelta(days=session_index, minutes=game_index)).isoformat(),\n                    "performance": event,\n                    "condition": condition,\n                    "applied_adjustment": next_level - level,\n                    "target_adj": target,\n                    "provenance": "synthetic-demo-policy-v1",\n                })\n                history.append({"accuracy": accuracy})\n                level = next_level\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(rows, indent=2))\n    print(f"Wrote {len(rows)} synthetic demo rows to {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/train-dda.py': '#!/usr/bin/env python3\n"""Offline training/export using the notebook\'s final RF pipeline and corrected history.\n\nInput JSON rows: patient_id, game_id, timestamp, performance (standard event),\ncondition, applied_adjustment (real prior action), target_adj (observed label).\nNo notebook execution and no generated/fabricated patient features.\nRun using the optional Python 3.12 DDA environment, from the repository root.\n"""\n\nimport argparse\nimport json\nimport statistics\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "backend"))\nfrom apps.games.notebook_dda import FEATURE_COLUMNS, FEATURE_CONTRACT, build_features\n\n\ndef main():\n    import joblib\n    import numpy as np\n    import pandas as pd\n    import sklearn\n    from sklearn.compose import ColumnTransformer\n    from sklearn.ensemble import RandomForestClassifier\n    from sklearn.metrics import f1_score\n    from sklearn.model_selection import GroupShuffleSplit\n    from sklearn.pipeline import Pipeline\n    from sklearn.preprocessing import OneHotEncoder\n\n    parser = argparse.ArgumentParser()\n    parser.add_argument("dataset", type=Path)\n    parser.add_argument("output", type=Path)\n    parser.add_argument("--version", required=True)\n    args = parser.parse_args()\n    rows = json.loads(args.dataset.read_text())\n    groups = [row["patient_id"] for row in rows]\n    train, test = next(\n        GroupShuffleSplit(test_size=0.2, random_state=42).split(rows, groups=groups)\n    )\n    # Fit the missing-history baseline on training users only, before feature construction.\n    neutral = statistics.median(\n        rows[index]["performance"]["reaction_time_ms"] / 1000 for index in train\n    )\n    histories = {}\n    features = [None] * len(rows)\n    for index in sorted(range(len(rows)), key=lambda i: rows[i]["timestamp"]):\n        row = rows[index]\n        if row["target_adj"] not in [-1, 0, 1] or row["applied_adjustment"] not in [\n            -1,\n            0,\n            1,\n        ]:\n            raise ValueError("Labels/actions must be -1, 0, +1")\n        current = row["performance"]\n        history = histories.setdefault((row["patient_id"], row["game_id"]), [])\n        features[index] = build_features(current, history, neutral, row["condition"])\n        history.append(\n            {\n                "accuracy": current["accuracy"],\n                "response_time": current["reaction_time_ms"] / 1000,\n                "hints_used": current["hints_used"],\n                "rounds": current["rounds_completed"],\n                "early_exit": current["early_exit"],\n                "adjustment": row["applied_adjustment"],\n            }\n        )\n    frame = pd.DataFrame(features, columns=FEATURE_COLUMNS)\n    labels = np.array([row["target_adj"] for row in rows])\n    if set(labels[train]) != {-1, 0, 1}:\n        raise ValueError("Training users must contain all three classes")\n    pipeline = Pipeline(\n        [\n            (\n                "prep",\n                ColumnTransformer(\n                    [\n                        (\n                            "cat",\n                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),\n                            ["condition"],\n                        ),\n                        ("num", "passthrough", FEATURE_COLUMNS[:-1]),\n                    ],\n                    verbose_feature_names_out=False,\n                ),\n            ),\n            (\n                "clf",\n                RandomForestClassifier(\n                    n_estimators=200,\n                    random_state=42,\n                    class_weight="balanced_subsample",\n                    n_jobs=-1,\n                ),\n            ),\n        ]\n    )\n    pipeline.fit(frame.iloc[train], labels[train])\n    prediction = pipeline.predict(frame.iloc[test])\n    # Bounds are learned only on training users; accuracy/difficulty remain exact contracts.\n    schema = {\n        key: {\n            "min": min(0, float(frame.iloc[train][key].min())),\n            "max": max(1, float(frame.iloc[train][key].max()) * 1.1),\n        }\n        for key in FEATURE_COLUMNS[:-1]\n    }\n    schema["accuracy"] = {"min": 0, "max": 1}\n    schema["current_difficulty"] = {"min": 0.1, "max": 1}\n    metadata = {\n        "model_version": args.version,\n        "feature_contract": FEATURE_CONTRACT,\n        "feature_columns": FEATURE_COLUMNS,\n        "feature_schema": schema,\n        "rt_neutral": neutral,\n        "library_versions": {\n            "sklearn": sklearn.__version__,\n            "pandas": pd.__version__,\n            "numpy": np.__version__,\n        },\n        "metrics": {\n            "held_out_user_macro_f1": f1_score(\n                labels[test], prediction, average="macro"\n            )\n        },\n        "limitations": "Engagement prototype; dataset provenance and supervision require review.",\n    }\n    joblib.dump({"pipeline": pipeline, "metadata": metadata}, args.output, compress=3)\n    cases = frame.iloc[test[:5]]\n    reloaded = joblib.load(args.output)["pipeline"]\n    assert np.array_equal(pipeline.predict(cases), reloaded.predict(cases))\n    args.output.with_suffix(".parity.json").write_text(\n        json.dumps(\n            [\n                {"features": features, "prediction": int(pred)}\n                for features, pred in zip(\n                    cases.to_dict("records"), reloaded.predict(cases), strict=True\n                )\n            ],\n            indent=2,\n        )\n    )\n    print(json.dumps(metadata, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, source in SOURCES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
for name in ["backend/apps/__init__.py", "backend/apps/games/__init__.py"]:
    (ROOT / name).touch()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("Embedded training sources prepared; no private GitLab token required.")


In [ ]:
subprocess.run([sys.executable, "scripts/generate-synthetic-dda.py", "artifacts/synthetic-dda-events.json", "--patients", "120", "--sessions-per-game", "16", "--seed", "42"], check=True)
rows = json.loads(Path("artifacts/synthetic-dda-events.json").read_text())
from collections import Counter
print("Rows:", len(rows), "Simulated users:", len({r["patient_id"] for r in rows}))
print("Classes:", dict(Counter(r["target_adj"] for r in rows)))
print("All rows synthetic:", all(r["provenance"] == "synthetic-demo-policy-v1" for r in rows))


In [ ]:
result = subprocess.run([sys.executable, "scripts/train-dda.py", "artifacts/synthetic-dda-events.json", "artifacts/dda-synthetic-demo-v1.joblib", "--version", "synthetic-demo-cloud-v1"], check=True, capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)


In [ ]:
import joblib, numpy as np, pandas as pd, sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score
sys.path.insert(0, str(ROOT / "backend"))
from apps.games.notebook_dda import build_features, FEATURE_COLUMNS, recommend
artifact_path = ARTIFACTS / "dda-synthetic-demo-v1.joblib"
artifact = joblib.load(artifact_path)
train, test = next(GroupShuffleSplit(test_size=0.2, random_state=42).split(rows, groups=[r["patient_id"] for r in rows]))
train_users = {rows[int(i)]["patient_id"] for i in train}
test_users = {rows[int(i)]["patient_id"] for i in test}
assert train_users.isdisjoint(test_users)
histories = {}
features = [None] * len(rows)
prior = [None] * len(rows)
for i in sorted(range(len(rows)), key=lambda i: rows[i]["timestamp"]):
    row = rows[i]
    current = row["performance"]
    history = histories.setdefault((row["patient_id"], row["game_id"]), [])
    prior[i] = list(history)
    features[i] = build_features(current, history, artifact["metadata"]["rt_neutral"], row["condition"])
    history.append({"accuracy":current["accuracy"], "response_time":current["reaction_time_ms"]/1000, "hints_used":current["hints_used"], "rounds":current["rounds_completed"], "early_exit":current["early_exit"], "adjustment":row["applied_adjustment"]})
frame = pd.DataFrame(features, columns=FEATURE_COLUMNS)
y = np.array([r["target_adj"] for r in rows])
pred = artifact["pipeline"].predict(frame.iloc[test])
parity = json.loads(artifact_path.with_suffix(".parity.json").read_text())
assert artifact["pipeline"].predict(pd.DataFrame([c["features"] for c in parity], columns=FEATURE_COLUMNS)).tolist() == [c["prediction"] for c in parity]
example_i = next(int(i) for i in test if len(prior[int(i)]) >= 6)
live = recommend(str(artifact_path), rows[example_i]["performance"], prior[example_i], rows[example_i]["condition"])
assert live["adjustment"] in [-1, 0, 1]
assert live["reason"] not in ["model_unavailable", "cold_start"]
report = {"repository":"https://gitlab.com/smarana-group1/smarana", "source_commit":'5d6dff35bbff0a4792e15137b620b07e04bdb62d', "provenance":"synthetic-demo-policy-v1", "seed":42, "rows":len(rows), "train_users":len(train_users), "test_users":len(test_users), "user_overlap":len(train_users & test_users), "held_out_user_macro_f1":float(f1_score(y[test], pred, average="macro")), "classification_report":classification_report(y[test], pred, labels=[-1,0,1], output_dict=True, zero_division=0), "confusion_matrix_labels":[-1,0,1], "confusion_matrix":confusion_matrix(y[test], pred, labels=[-1,0,1]).tolist(), "reload_parity_passed":True, "runtime_example":live, "artifact_sha256":hashlib.sha256(artifact_path.read_bytes()).hexdigest(), "source_sha256":{name:hashlib.sha256(source.encode()).hexdigest() for name,source in SOURCES.items()}, "limitations":"Synthetic policy imitation only; no real patients or clinical validation. Keep separate from clinical deployment."}
(ARTIFACTS / "training-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))


In [ ]:
import zipfile
bundle = ROOT / "smarana-dda-synthetic-demo-cloud-v1.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(ARTIFACTS.glob("*")):
        if path.name != "synthetic-dda-events.json": z.write(path, "artifacts/" + path.name)
    for name in SOURCES: z.write(ROOT/name, "sources/"+name)
print("Export ready:", bundle, "bytes:", bundle.stat().st_size)
from google.colab import files
files.download(str(bundle))


## Deployment boundary

Install the same pinned ML library versions on the demo server. Extract the artifact and set `DDA_MODEL_ARTIFACT` to its absolute path only for the demo. Existing safety wrappers and deterministic fallback remain in place. Real-world use requires reviewed labels, de-identified real data, and separate validation. The ZIP includes measured evaluation, library/feature metadata, source hashes, and export-parity cases.
